# YOLO 청구기호 라벨 검출기 **v5** — 박스 기준 완화 + 하드 네거티브

**v3에서 바뀐 것:** ①학습 박스를 하단 40%→54%+옆 8%(앱 크롭 패딩 기하와 일치)로 완화 —
타이트 박스가 분류번호 줄을 자르던 문제의 근본 수정 ②사람이 '라벨아님' 확정한 오탐 크롭
17건을 배경 이미지로 투입. v4(부트스트랩)는 '검출 무죄' 판명으로 폐기, 승격 없음.
나머지(증강·epochs·모델)는 v3와 동일 — 변인 통제.

**데이터:** `yolo_labelset_v5.zip` (train 172+배경 17 · val 34)
**채택 기준(로컬 A/B):** 걷기 표본 검출 수 v3 이상 + 사진 회귀 회복 + 웹데모 autotest 14권 유지
**런타임:** GPU(T4) · 약 40~50분


In [ ]:
# 1) 설치 + 데이터 업로드 (yolo_labelset_v5.zip)
!pip install -q ultralytics
from google.colab import files
up = files.upload()   # yolo_labelset_v5.zip
!unzip -oq yolo_labelset_v5.zip
!ls yolo_labelset_v5/images/train | wc -l; ls yolo_labelset_v5/images/val | wc -l


In [ ]:
# 2) 학습 — v3와 동일 하이퍼파라미터 (변인은 데이터셋뿐)
from ultralytics import YOLO
try:
    model = YOLO('yolo26n.pt')
except Exception as e:
    print('yolo26n 불가 → yolo11n 폴백:', e)
    model = YOLO('yolo11n.pt')
model.train(data='yolo_labelset_v5/data.yaml', epochs=120, imgsz=1280, batch=8,
            degrees=12, perspective=0.0008, shear=4, fliplr=0.0,
            hsv_v=0.5, hsv_s=0.4, mosaic=0.6, patience=30, name='call_label_v5')


In [ ]:
# 3) 검증 수치 + 시각 확인
m = YOLO('runs/detect/call_label_v5/weights/best.pt')
r = m.val(data='yolo_labelset_v5/data.yaml', imgsz=1280)
print('mAP50:', r.box.map50, 'mAP50-95:', r.box.map)
import glob
res = m.predict(glob.glob('yolo_labelset_v5/images/val/*.jpg')[0], imgsz=1280, conf=0.3, save=True)
print('예측 저장:', res[0].save_dir)


In [ ]:
# 4) ONNX 내보내기 + 다운로드
m.export(format='onnx', imgsz=1280, half=False, dynamic=False)
!zip -q -j call_label_yolo_v5.zip runs/detect/call_label_v5/weights/best.pt runs/detect/call_label_v5/weights/best.onnx
from google.colab import files
files.download('call_label_yolo_v5.zip')


## 5) 로컬 채점 (다운로드 후)

`call_label_yolo_v5.zip`을 저(클로드)에게 주시면:
① 동적 ONNX 재수출(라이브 960/사진 1280 겸용) ② 걷기 표본·사진 회귀 A/B(v3 대비)
③ 웹데모 autotest(14권 유지) + 패딩 조정 영향 분석 → **보고 후 승인받고 배포**
